# Бінарны класіфікатар на градыентным бустынгу

Прадказвацьмем лагарытм шанцаў (log-odds), які канвертавацьмем у імавернасці праз сігмойду, як у лагістычнай рэгрэсіі.<br>Датасэт для прадказанняў — добра знаёмы Breast Canser.

In [75]:
import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeRegressor as DTR
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score as Acc, roc_auc_score as ROC_AUC
from sklearn.ensemble import GradientBoostingClassifier as GBC

#### Сігмойда

In [76]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

## Клас GBMClassifier

*Cэмпляванне радкоў, падтрымка катэгарыйных прызнакаў супадае з рэалізацыяй для задачы рэгрэсіі.*

### Канструктар

In [77]:
class GBMClassifier:
    def __init__(self, n=100, lr=0.1, depth=3, subsample=1.0, colsample_bytree=1.0, seed=42):
        self.n = n # колькасць дрэваў
        self.lr = lr # learning rate
        self.depth = depth # максімальная глыбіня дрэва
        self.subsample = subsample # доля радкоў (аб'ектаў) для трэніроўкі 1 дрэва
        self.colsample_bytree = colsample_bytree # доля калонак (прызнакаў) для трэніроўкі 1 дрэва
        self.seed = seed
        self.trees = [] # спіс трэніраваных дрэваў
        self.trees_features = [] # індэксы прызнакаў для кожнага дрэва
        self.initial_log_odds = 0.0 # стартавае прадказанне

### Функцыя трэніроўкі

Звесткі будзем захоўваць у якасці NumPy-масіваў дзеля хуткасці.

**Крок I.<br>**
Для стартавага прадказання возьмем сярэдняе значэнне ў $y$ ў якасці шанца класа 1. <br>Каб пазбегнуць дзялення на нуль і лагарытму адзінкі, выкарыстаем `numpy.clip()`.<br>Палічым лагарытм адносінаў шанцаў класа 1 да класа 0 ($\log{\frac{P_1}{P_0}}$).

##### *Наступныя крокі паўтараюцца для кожнага дрэва:*
**Крок II.<br>**
Сырыя прадказанні сігмойда пераводзіць у імавернасці.

**Крок III.<br>**
Нашая верагоднасць — сігмойда (тут `raw_predictions` азначаная $r$):
$$
p = \sigma(r) = \frac{1}{1 + e^{-r}}
$$

Log-loss:
$$
L(r) = -(y\log(p(r)) + (1-y)(log(1-p(r))))
$$

Можна доўга й мутарна лічыць яе вытворную, але спойлер: $\frac{\partial L}{\partial p} = p - y$, а значыць, антыградыент $y - p$

**Крок IV.<br>**
Як і для рэгрэсіі, абярэм выпадковыя радкі й калонкі, захаваўшы выкарыстаныя для кожнага дрэва прызнакі. 

**Крок V.<br>**
Навучым на абраных прызнаках і аб'ектах дрэва й абновім прадказанні.


In [78]:
def GBMC_fit(self, X, y):
    X_arr = np.array(X)
    y_arr = np.array(y)

    p1 = np.mean(y_arr)
    p1 = np.clip(p1, 1e-15, 1 - 1e-15)
    self.initial_log_odds = np.log(p1 / (1 - p1))

    raw_predictions = np.full(len(y_arr), self.initial_log_odds)

    rng = np.random.default_rng(self.seed)

    for i in range(self.n):
        probs = sigmoid(raw_predictions)

        antigrad = y_arr - probs

        n_rows, n_cols = X_arr.shape
        row_idx = rng.choice(n_rows, size=int(n_rows * self.subsample), replace=False)
        col_idx = rng.choice(n_cols, size=max(1, int(n_cols * self.colsample_bytree)), replace=False)
        self.trees_features.append(col_idx)

        tree = DTR(max_depth=self.depth, random_state=rng.integers(10000))
        tree.fit(X_arr[row_idx][:, col_idx], antigrad[row_idx])
        self.trees.append(tree)
        raw_predictions += tree.predict(X_arr[:, col_idx]) * self.lr

GBMClassifier.fit = GBMC_fit

## Функцыя прадказання ймавернасцяў

In [79]:
def GBMC_predict_proba(self, X):
    X_arr = np.array(X)

    raw_predictions = np.full(len(X_arr), self.initial_log_odds)

    for tree, col_idx in zip(self.trees, self.trees_features):
            raw_predictions += self.lr * tree.predict(X_arr[:, col_idx])

    probabilities = sigmoid(raw_predictions)

    return np.vstack((1 - probabilities, probabilities)).T

GBMClassifier.predict_proba = GBMC_predict_proba

## Функцыя прадказання класаў

In [80]:
def GBMC_predict(self, X):
        probabilities = self.predict_proba(X)[:, 1]
        return (probabilities >= 0.5).astype(int)

GBMClassifier.predict = GBMC_predict

# Тэст мадэлі

## Падрыхтоўка

In [81]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Гіперпараметры

In [82]:
N_TREES = 150
LR = 0.1
DEPTH = 3
SUBSAMPLE = 0.8

## Напісаная мадэль GMBClassifier

In [83]:
my_gbm = GBMClassifier(n=N_TREES, lr=LR, depth=DEPTH, subsample=SUBSAMPLE, colsample_bytree=1.0, seed=42)
my_gbm.fit(X_train, y_train)

my_preds = my_gbm.predict(X_test)
my_probas = my_gbm.predict_proba(X_test)[:, 1]

my_acc = Acc(y_test, my_preds)
my_roc_auc = ROC_AUC(y_test, my_probas)

## GradientBoostingClassifier *ад* scikit-learn

In [84]:
sk_gbm = GBC(n_estimators=N_TREES, learning_rate=LR, max_depth=DEPTH, subsample=SUBSAMPLE, random_state=42)
sk_gbm.fit(X_train, y_train)

sk_preds = sk_gbm.predict(X_test)
sk_probas = sk_gbm.predict_proba(X_test)[:, 1]

sk_acc = Acc(y_test, sk_preds)
sk_roc_auc = ROC_AUC(y_test, sk_probas)

## Параўнанне

In [85]:
df = pd.DataFrame({
    "Metric": ["Accuracy", "ROC-AUC"],
    "Mая GBM": [my_acc, my_roc_auc],
    "Sklearn GBM": [sk_acc, sk_roc_auc]
})

df = df.set_index("Metric")
df.index.name = None

df.round(4)

,Mая GBM,Sklearn GBM
Accuracy,0.9561,0.9649
ROC-AUC,0.9934,0.9941
